# SIR Epidemic Model

*System Dynamics Studio, model library: Complex behavior*

The SIR model is one of the most famous compartmental models in epidemiology. It divides a population into three groups: Susceptible (can catch the disease), Infected (have the disease and can spread it), and Recovered (cannot catch it again). This model helps scientists predict how fast an outbreak will spread, what the peak number of infections will be, and how "herd immunity" eventually stops the transmission.

**Concepts:** Conservation, Nonlinear contact, Infection peak

**Expected behavior:** Infected rises to a single peak, then falls as susceptibles are depleted. S + I + R stays constant.

**Limitations:** Closed population (no births, deaths or waning immunity); homogeneous mixing.

## Model structure

**Stock balances**

$$
\begin{aligned}
\frac{d\,\text{Susceptible}}{dt} &= - \text{infection\_rate} \\
\frac{d\,\text{Infected}}{dt} &= \text{infection\_rate} - \text{recovery\_rate} \\
\frac{d\,\text{Recovered}}{dt} &= \text{recovery\_rate} \\
\end{aligned}
$$

**Stocks**

| Stock | Initial value | Unit |
|---|---|---|
| `Susceptible` | 990 | — |
| `Infected` | 10 | — |
| `Recovered` | 0 | — |

**Flows and auxiliaries** (in evaluation order)

$$
\begin{aligned}
\text{infection\_rate} &= \frac{\text{Susceptible}\cdot\text{Infected}\cdot\text{infection\_probability}}{\text{total\_pop}} \\
\text{recovery\_rate} &= \frac{\text{Infected}}{\text{recovery\_delay}} \\
\end{aligned}
$$

**Parameters**

| Parameter | Value | Unit |
|---|---|---|
| `total_pop` | 1000 | — |
| `infection_probability` | 0.4 | — |
| `recovery_delay` | 10 | — |

## Setup

NumPy, SciPy and Matplotlib are preinstalled on Google Colab. The step, ramp and pulse helpers
are the time functions available in the app; they are defined here for completeness.

In [ ]:
# Generated by System Dynamics Studio
# Model time: 0 to 120 (time unit not set), output every 0.1
# In-app solver: RK4, fixed dt = 0.1.
# This script integrates with scipy.integrate.odeint (adaptive LSODA), so values
# match the app within numerical tolerance, not digit for digit.
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Time functions (discontinuous; same definitions as in the app)
def sd_step(height, start, t):
    return np.where(t >= start, height, 0.0)

def sd_ramp(slope, start, *rest):
    *end, t = rest
    until = np.minimum(t, end[0]) if end else t
    return np.where(t > start, slope * (until - start), 0.0)

def sd_pulse(height, start, width, t):
    return np.where((t >= start) & (t < start + width), height, 0.0)

## Parameters and initial values

Edit the values below and rerun the notebook from here to try your own scenarios.

In [ ]:
# 1. Model Parameters (Constants)
params = {
    'total_pop': 1000,
    'infection_probability': 0.4,
    'recovery_delay': 10
}

# 2. Initial State (Stocks)
y0 = [
    990,  # Susceptible
    10,  # Infected
    0  # Recovered
]

## Model equations

`model` returns the rate of change of every stock. It is the system of ordinary differential equations shown above.

In [ ]:
# 3. Model Definition
def model(y, t, p):
    # Unpack stocks
    Susceptible, Infected, Recovered = y

    # Auxiliaries & Flows (dependency order)
    infection_rate = (((Susceptible * Infected) * p['infection_probability']) / p['total_pop'])
    recovery_rate = (Infected / p['recovery_delay'])

    # Derivatives (Change in Stocks)
    dSusceptible_dt = - infection_rate
    dInfected_dt = infection_rate - recovery_rate
    dRecovered_dt = recovery_rate

    return [dSusceptible_dt, dInfected_dt, dRecovered_dt]

## Simulation

The model runs from t = 0 to t = 120 time units. The app integrates with RK4 at a fixed
dt = 0.1; this notebook uses SciPy's adaptive `odeint` (LSODA) with tight tolerances, so results agree
with the app within numerical tolerance.

In [ ]:
# 4. Run Simulation
t = np.linspace(0, 120, 1201)
sol = odeint(model, y0, t, args=(params,), rtol=1e-8, atol=1e-10)

In [ ]:
# 5. Post-Processing (Calculate variables for plotting)
Susceptible = sol[:, 0]
Infected = sol[:, 1]
Recovered = sol[:, 2]
total_pop = np.full_like(t, params['total_pop'])
infection_probability = np.full_like(t, params['infection_probability'])
recovery_delay = np.full_like(t, params['recovery_delay'])
infection_rate = (((Susceptible * Infected) * params['infection_probability']) / params['total_pop']) + np.zeros_like(t)
recovery_rate = (Infected / params['recovery_delay']) + np.zeros_like(t)

## Results

In [ ]:
# 6. Visualization
plt.figure(figsize=(10, 6))
plt.plot(t, Susceptible, label='Susceptible', linewidth=2)
plt.plot(t, Infected, label='Infected', linewidth=2)
plt.plot(t, Recovered, label='Recovered', linewidth=2)

plt.title('System Dynamics Simulation (Python)')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment

Halving the infection probability: is the epidemic peak lower, later, or both?

The cell below reruns the model with one change and compares each stock against the baseline.

In [ ]:
STOCKS = ['Susceptible', 'Infected', 'Recovered']

def simulate(param_changes=None, initial_changes=None):
    """Rerun the model with changed parameters and/or initial stock values."""
    p = {**params, **(param_changes or {})}
    y_init = list(y0)
    for stock, value in (initial_changes or {}).items():
        y_init[STOCKS.index(stock)] = value
    return odeint(model, y_init, t, args=(p,), rtol=1e-8, atol=1e-10)

scenario = simulate(param_changes={'infection_probability': 0.2}, initial_changes={})

fig, axes = plt.subplots(len(STOCKS), 1, sharex=True, figsize=(10, 2.8 * len(STOCKS)))
axes = np.atleast_1d(axes)
for i, (ax, stock) in enumerate(zip(axes, STOCKS)):
    ax.plot(t, sol[:, i], label='Baseline', linewidth=2)
    ax.plot(t, scenario[:, i], '--', label='infection_probability = 0.2', linewidth=2)
    ax.set_ylabel(stock)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='best')
axes[-1].set_xlabel('Time')
fig.suptitle('Baseline vs. scenario')
plt.tight_layout()
plt.show()

## Questions to explore

- Lower infection_probability. Does the peak get lower, later, or both?
- Check that S + I + R stays at 1000 in the data table.